# STAT 764 · Meeting 18 — The bake-off

**Thursday, November 5**

Three meetings built three flexible models: a tree, a forest, a boosted model. Today they meet Meeting 9's
logistic regression and a **foundation model**, TabICL, trained in advance on other tables, which needs no
tuning at all. Same patients, same test set.

**The question for today.** When several models score about the same, what does "better" mean? We judge
them six ways: performance, calibration, stability, tuning burden, cost and interpretability. Then you
recommend one for the \$150 home visit from Meeting 10: a nurse visits a patient when the predicted
chance of readmission is at least 0.169, the break-even for a visit that costs \$150 and prevents 5% of a
\$17,700 readmission.

The page `bakeoff.html` in this folder has the same ideas with pictures. Open it in a web browser.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import time

import numpy as np
import pandas as pd
from scipy.special import logit
from stat764 import diabetes, load

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The readmission data from Meetings 9-12, with Meeting 9's split by PATIENT (split 764).
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))
X_test, y_test = X_all.iloc[test], y_all.iloc[test].to_numpy()
test_patients = enc["patient_nbr"].iloc[test].to_numpy()    # which patient each test stay is


def sample_rows(seed):
    """A random 15,000 of the training PATIENTS, all their stays. Seed 16 is Meeting 16's sample."""
    pool = enc["patient_nbr"].iloc[train].unique()
    chosen = np.random.default_rng(seed).choice(pool, 15_000, replace=False)
    return train[np.isin(enc["patient_nbr"].iloc[train], chosen)]


def prep():
    """Meeting 9's preprocessing: fill and scale the counts, one-hot the categories."""
    return ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
    ])


def make_pipe(model):
    """Preprocessing inside the pipeline, any model at the end."""
    return Pipeline([("prep", prep()), ("model", model)])


# The three models we fit, each with the settings it earned in its own meeting.
MODELS = {
    "logistic regression": lambda: LogisticRegression(max_iter=2000),                     # Meeting 9
    "random forest": lambda: RandomForestClassifier(n_estimators=200, max_features="sqrt",
                                                    min_samples_leaf=5, n_jobs=-1, random_state=0),  # Meeting 16
    "boosting": lambda: GradientBoostingClassifier(n_estimators=34, learning_rate=0.1, max_depth=3,
                                                   random_state=0),                       # Meeting 17
}

# TabICL's predictions for the test stays, one column per training sample. They were computed in advance
# on a GPU: it needs one, and it is not in the course environment. See "See it in Scikit-Learn" on the page.
tabicl = load("tabicl_test_predictions.csv")

READMISSION = 17_700                                   # dollars, Meeting 10
P_STAR = 150 / (0.05 * READMISSION)                    # the $150 visit's break-even: 0.169


def net_per_1000(outcome, probs):
    """Meeting 10's money: dollars saved minus dollars spent per 1,000 discharges, visiting probs >= P_STAR."""
    visit = probs >= P_STAR
    return ((visit & (outcome == 1)).sum() * 0.05 * READMISSION - visit.sum() * 150) / len(outcome) * 1000


def calibration_slope(probs, outcome):
    """Meeting 11's slope: 1 = the spread is right; below 1 = too extreme; above 1 = too timid."""
    z = logit(np.clip(probs, 1e-6, 1 - 1e-6)).reshape(-1, 1)
    return LogisticRegression(C=1e6, max_iter=1000).fit(z, outcome).coef_[0, 0]


def scores(probs, outcome=y_test):
    """One row of the scoreboard: the order, the numbers, and the money."""
    return {"AUC": roc_auc_score(outcome, probs), "Brier": brier_score_loss(outcome, probs),
            "average said": probs.mean(), "calibration slope": calibration_slope(probs, outcome),
            "share visited": np.mean(probs >= P_STAR), "net per 1,000": net_per_1000(outcome, probs)}


print(f"test: {len(y_test):,} stays · came back {y_test.mean():.1%} · break-even {P_STAR:.3f}")

## 1. Four models, same patients

Every model below learns from Meeting 16's 15,000 training patients and is scored on the same test stays.
The first three take about a second each here; TabICL's column was loaded from the file.

In [ ]:
probs, seconds = {}, {}
rows = sample_rows(16)
for name, make in MODELS.items():
    t0 = time.time()
    pipe = make_pipe(make()).fit(X_all.iloc[rows], y_all.iloc[rows].to_numpy())
    probs[name] = pipe.predict_proba(X_test)[:, 1]
    seconds[name] = time.time() - t0
probs["TabICL"] = tabicl["sample1"].to_numpy()

board = pd.DataFrame({name: scores(p) for name, p in probs.items()}).T
print(board.round(4).to_string())
print(f"\ncame back on test: {y_test.mean():.4f}")
print("seconds to fit and predict here:", {k: round(v, 1) for k, v in seconds.items()})

TabICL ranks best, and it was never tuned. But the table has more than one winner: its probabilities run
low (it says 9.8% when 11.1% came back), the forest's slope is the closest to 1, and the money does not
simply follow the AUC. One test set, though, is one draw. Are these gaps real?

## 2. Is a gap real? A paired bootstrap

The AUC of two models on one test set differs partly because of **which patients** happen to be in it. To
see how much, redraw the test patients with replacement, score **both** models on each redraw, and keep the
difference. Three things make this the right check:

- **Resample patients, not stays,** because a patient's stays go together (Meeting 9).
- **Keep the pair.** Both models face the same redraw, so the luck they share cancels. The difference is
  much steadier than either score alone.
- **Read the interval.** If the middle 95% of the differences includes 0, this test set cannot tell the two
  models apart.

In [ ]:
_ids, _which = np.unique(test_patients, return_inverse=True)
_members = np.split(np.argsort(_which), np.cumsum(np.bincount(_which))[:-1])   # each patient's test rows


def paired_gap(p, q, measure=roc_auc_score, draws=300, seed=764):
    """measure(p) - measure(q) on the test stays, and the middle 95% over redraws of the test PATIENTS."""
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(draws):
        b = np.concatenate([_members[i] for i in rng.integers(0, len(_ids), len(_ids))])
        diffs.append(measure(y_test[b], p[b]) - measure(y_test[b], q[b]))
    return measure(y_test, p) - measure(y_test, q), *np.percentile(diffs, [2.5, 97.5])


gap, lo, hi = paired_gap(probs["random forest"], probs["logistic regression"])
print(f"forest minus logistic regression, AUC: {gap:+.4f}  (middle 95% of redraws: {lo:+.4f} to {hi:+.4f})")

## Studio

**Everything you need is in this notebook.** Sections 1 and 2 above explain each idea, and each question
below lists its steps. The page `bakeoff.html` in this folder walks through the same ideas with pictures.

| | what you practice | about |
|---|---|---|
| **Question 1** | deciding which gaps are real, in AUC and in dollars | 15 minutes |
| **Question 2** | asking whether a model would say the same thing next time | 15 minutes |
| **Question 3** | recommending one model, and defending it with numbers | 15 minutes |

**All three questions count.** If you do not finish in class, finish at home: the notebook is due 24
hours after class.

**How to work.**

1. If you opened this notebook fresh, run everything above first (**Run All**). The questions use `probs`,
   `board`, `paired_gap`, `net_per_1000`, `sample_rows`, `MODELS`, `make_pipe` and `tabicl`.
2. Before each question, **write your guess** as a comment in its cell, and keep it when the answer
   disagrees. The Compare uses it.
3. Each question ends with sentences to answer. Write them in a comment or a markdown cell under its output.
4. Save and upload to Canvas under "Studio notebook — Day 18" by **Friday, November 6, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `NameError` for `probs`, `paired_gap` or `tabicl` | Run the notebook from the top (Run All). |
| `FileNotFoundError` or a network error for `tabicl_test_predictions.csv` | Run `git pull` in your clone: the file came with today's update. |
| `paired_gap` is slow | Each call redraws the patients 300 times: about 10 seconds. That is normal. |
| Every interval is huge, or `TypeError` in `paired_gap` | `measure` takes `(outcome, probs)`, in that order, like `roc_auc_score` and `net_per_1000`. |
| Question 2: the overlap is 1.0 | You compared a sample with itself. Compare two different samples. |

### 1. Which gaps are real?

- **(a)** For the forest, boosting and TabICL, `paired_gap(probs[name], probs["logistic regression"])`: the
  difference in AUC from logistic regression, with its interval.
- **(b)** The same in dollars: `paired_gap(probs[name], probs["logistic regression"], measure=net_per_1000)`.
- **(c)** Put all six in one table.

**Answer in a sentence or two:** which gaps can this test set tell from zero, in AUC and in dollars? Does
the best AUC come with clearly more money?

### 2. Would it say the same thing next time?

A hospital retrains its model. Would the same patients get visited?

- **(a)** For logistic regression, the forest and boosting, fit on five samples of 15,000 patients, seeds
  `(16, 101, 102, 103, 104)`, using `sample_rows(seed)` and each model in `MODELS`, and keep the test
  probabilities (`ps[name]` a list of five arrays). TabICL's five are `tabicl[f"sample{k}"]`, k = 1 to 5.
- **(b)** For each model, across the five samples: the smallest and largest AUC.
- **(c)** Take samples 1 and 2. **How far does one stay's prediction move?** `np.mean(np.abs(ps[name][0] -
  ps[name][1]))`. **How much do the top-10% lists overlap?** One way:

  ```python
  k = len(y_test) // 10
  top = lambda p: set(np.argsort(-p)[:k])
  overlap = len(top(a) & top(b)) / k
  ```

**Answer in a sentence or two:** which model changes its mind the most about individual patients, and
does that show up in its AUC?

### 3. Your recommendation

The hospital's program pays for a \$150 home visit at the 0.169 break-even. You have your scoreboard, your
intervals and your stability numbers. Two axes are not in a table:

- **Tuning burden and cost:** logistic regression has almost nothing to tune; the forest and boosting took
  settings earned in Meetings 16 and 17; TabICL has no tuning but needs a GPU to be quick (about 4 seconds
  here; about 5 minutes on 24 CPU cores).
- **Interpretability:** what can you tell a nurse about one patient's score? Logistic regression has
  coefficients; a forest or boosted model needs tools such as permutation importance; TabICL offers neither.

Fill in the `scorecard` dictionary with one number or a few words per axis for the model you choose.

**Answer in four to six sentences:** which model would you deploy, and why? Name a number for each axis
that decided it, and the one result that would change your mind.

In [ ]:
# Studio setup for question 3 -- fill this in once you have chosen.
scorecard = {"model": "", "performance": "", "calibration": "", "stability": "", "tuning": "", "cost": "",
             "interpretability": ""}

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of next class

We compare answers at the start of next class (Tuesday, November 10), built from the notebooks you upload.
So leave these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: your table of gaps and intervals.
3. Question 3: your recommendation.

## Exit ticket

> A model with no tuning did about as well as the ones you tuned. In two or three sentences: what did your
> weeks of pipeline work buy you?

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio notebook —
Day 18". Your two studio points need this answered: a blank exit ticket earns zero. It is not graded for
being right — it is how I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **Grinsztajn, Oyallon & Varoquaux (2022)** — this week's paper: why tree-based models still beat neural
networks on tables like ours, with a fair tuning budget for every model.

📗 **ESL §10.12** — regularization in boosting: shrinkage and subsampling.

🎓 **Meeting 19 (Tuesday)** — interpretation: what a model's explanations can and cannot tell you.